In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
from SamplingMethods import Sampler_class

In [2]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/ModelXG/ModelMk1.json")

In [3]:
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = loaded_model.predict(np.array([[s1],[s2],[b1]]).T)[0]
    return np.float64(y_pred)

In [4]:
class client(object):
    def __init__(self, df):
        self.df = df
    def summarize(self):
        return df

In [5]:
class RangeParameterConfig(object):
    def __init__(self, name, bounds):
        self.name = name
        self.bounds = bounds

In [6]:
class OptimisationSetup_class(object):
    def __init__(self):
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", bounds=(0, 1)),
            RangeParameterConfig(name="s2", bounds=(0, 1)),
            RangeParameterConfig(name="b1", bounds=(0, 1)),
        ]
OptimisationSetup_obj = OptimisationSetup_class()

In [7]:
y_max_lis = []

for i in range(100):
    sampler_obj = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler_obj.three.QuasirandomSampler3D_func(8,Parameters_lis).T
    y = []
    for row in X:
        y.append(SurrogateModelOfReality(row[0],row[1],row[2]))
    y = np.array(y)
    d = {"s1": X.T[0], "s2": X.T[1], "b1": X.T[2], "t1": y}
    df = pd.DataFrame(data=d)
    client_obj = client(df)
    # client_obj.summarize()
    sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client_obj)

    for _ in range(19):
        trial = sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client_obj)
        s1 = trial[0][0]
        s2 = trial[0][1]
        b1 = trial[0][2]
        result = SurrogateModelOfReality(s1,s2,b1)
        nd = {"s1": [s1], "s2": [s2], "b1": [b1], "t1": [result]}
        df_new_rows = pd.DataFrame(data=nd)
        df = pd.concat([df,df_new_rows],ignore_index=True)
        client_obj = client(df)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client_obj.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

Trial 0 =========================================
14.92771053314209

Trial 1 =========================================
14.701310157775879

Trial 2 =========================================
13.910550117492676

Trial 3 =========================================
14.027234077453613

Trial 4 =========================================
14.761591911315918

Trial 5 =========================================
15.040355682373047

Trial 6 =========================================
14.023553848266602

Trial 7 =========================================
14.442544937133789

Trial 8 =========================================
14.30508804321289

Trial 9 =========================================
14.337114334106445

Trial 10 =========================================
15.303513526916504

Trial 11 =========================================
13.823448181152344

Trial 12 =========================================
13.717066764831543

Trial 13 =========================================
15.125311851501465

Trial 14 =========

In [8]:
y_max_arr = np.array(y_max_lis)
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 15.939385414123535
Avg = 14.50393892288208
Std = 0.4929190837242422


In [9]:
print(y_max_arr.tolist())

[14.92771053314209, 14.701310157775879, 13.910550117492676, 14.027234077453613, 14.761591911315918, 15.040355682373047, 14.023553848266602, 14.442544937133789, 14.30508804321289, 14.337114334106445, 15.303513526916504, 13.823448181152344, 13.717066764831543, 15.125311851501465, 14.220800399780273, 14.94223690032959, 14.161872863769531, 14.2437744140625, 13.991470336914062, 15.283529281616211, 13.717000007629395, 14.353147506713867, 14.027234077453613, 15.039326667785645, 14.459877967834473, 14.091459274291992, 14.088593482971191, 14.231822967529297, 14.337114334106445, 14.118279457092285, 14.476275444030762, 15.13394546508789, 14.493083000183105, 14.12717342376709, 14.28302001953125, 14.691985130310059, 14.78408145904541, 14.2388277053833, 14.557159423828125, 14.640384674072266, 15.073307037353516, 14.87447738647461, 14.17615032196045, 14.44465160369873, 14.275142669677734, 14.495916366577148, 14.64218807220459, 14.161872863769531, 15.221745491027832, 14.790014266967773, 14.50677680969

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestsOfModelXGB/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [11]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestsOfModelXGB/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    14.654818
1    13.876438
2    13.796576
3    15.344110
4    14.092174
..         ...
495  15.732443
496  14.873192
497  15.469332
498  15.347440
499  13.912857

[500 rows x 1 columns]


In [12]:
# # Sanity check to make sure the MIPT is running correctly.
# df = client.summarize()
# types_lis = []
# for i in range(len(df)):
#     if i < 8:
#         types_lis.append("one-shot")
#     else:
#         types_lis.append("sequential")
# df["type"] = types_lis
# fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='type',width=1300, height=600)
# fig.show()